# {{来源}}数据清洗 — 逐类确认

这份 notebook 一节看一类数据问题，每节都是同一个看法：
**这个问题影响多少数据 → 集中在什么时候 → 具体是哪几笔**，最后给出处理建议。

- 处理建议只是建议，**怎么处理由你拍板**——写在每节末尾的「你的结论」里，定了再看下一类。
- 这里只做查看和试算，**不会改动任何原始数据**；确认过的结论会整理进方案文档。

In [ ]:
df = read("{{窗口}}")   # 例：read_inbound("2026-09")

pd.DataFrame([{
    "行数": f"{len(df):,}",
    "列数": df.shape[1],
    "金额合计(万)": round(float(df["{{金额列}}"].sum()), 1),
    "日期范围": f"{df['{{日期列}}'].min()} ~ {df['{{日期列}}'].max()}",
}])

## 一、先给每列做个体检

看看每一列缺了多少、有多少种取值。排在前面的列问题最大，也是后面逐节重点查的对象。

In [ ]:
# 每列体检：什么类型、缺了多少、有多少种取值；缺得最狠的排前面
profile = pd.DataFrame({
    "类型": df.dtypes.astype(str),
    "非空率": df.notna().mean().round(3),
    "取值种数": df.nunique(),
})
profile.sort_values(["非空率", "取值种数"]).head(15)

**体检结论**（跑完上面一格后回填）：{{哪些列有什么问题}}。下面按严重程度逐类来看。

---

## 二、{{异常类}}

**怎么认出来的**：{{用大白话说清什么样的数据算这类问题，点到具体字段}}

In [ ]:
# 这个问题影响多少数据
mask = {{判定表达式}}          # 例：df.duplicated(keep=False)

pd.DataFrame([{
    "问题行数": f"{int(mask.sum()):,}",
    "占比": f"{mask.mean():.1%}",
    "涉及金额(万)": round(float(df.loc[mask, "{{金额列}}"].sum()), 1),
}])

In [ ]:
# 问题集中在哪些月份
hit = df.loc[mask]
hit.groupby(hit["{{日期列}}"].dt.strftime("%Y-%m")).size().to_frame("问题行数")

In [ ]:
# 具体是哪几笔（随机抽 10 笔，带单号、物料等上下文，方便核对）
cols = ["{{单号列}}", "{{物料列}}", "{{日期列}}", "{{问题列}}", "{{金额列}}"]
hit[cols].sample(min(10, len(hit)), random_state=42)

**建议怎么处理**：{{修格式 / 这部分先不参与统计 / 保留但打上标记}} —— {{一句话理由}}

**你的结论**：{{同意 / 不同意，改成 ___}} —— 你定了我们再看下一类

---

{{「一节 + 四格」照这个样子按问题类型复制，编号往后排}}

## 最后、把确认过的处理都套上，整体跑一遍

（所有问题都确认完才跑这一节）跑完对一遍总账：
清洗前的数据 = 清洗后保留的 + 各类剔除的，对得上才算干净。

In [ ]:
# 把每一节确认过的处理套上去，整体跑一遍
clean = df.{{已确认处置链}}   # 例：.pipe(去重).pipe(合并退货单价)

audit = pd.DataFrame([
    {"项": "{{异常类1}} 剔除", "行数": {{n1}}, "金额(万)": {{m1}}},
    {"项": "{{异常类2}} 剔除", "行数": {{n2}}, "金额(万)": {{m2}}},
    {"项": "清洗前", "行数": len(df),    "金额(万)": round(float(df["{{金额列}}"].sum()), 1)},
    {"项": "清洗后", "行数": len(clean), "金额(万)": round(float(clean["{{金额列}}"].sum()), 1)},
])
audit   # 对总账：清洗前 = 清洗后 + 剔除合计；对不上就停下查

In [ ]:
# 清洗后的底子再体检一遍，确认可以进入指标计算
pd.DataFrame({
    "非空率": clean.notna().mean().round(3),
    "取值种数": clean.nunique(),
}).sort_values("非空率").head(10)

**还有待定的**：{{没拍板的问题，逐条列在这里}}。

这一版清洗结果先不落盘；等第四步做成正式脚本时，把这些确认过的规则一起固化成代码。